# 3.2 — Sentence Segmentation

**Natural Language Processing (NLP) | Python, spaCy, and NLTK**

This notebook introduces methods for splitting text into sentences, inspecting sentence boundaries, and adapting segmentation to specific needs.

## Learning objectives

By the end of this notebook, you should be able to:
- explain sentence segmentation;
- segment text with `spaCy` and `NLTK`;
- inspect sentence boundaries in `spaCy`;
- understand why punctuation alone is not always sufficient;
- test segmentation on English and Arabic text.

## 1. Environment setup

Run this cell if the libraries are not already installed. In Google Colab, you can run it directly.

In [ ]:
%pip install -q spacy nltk

## 2. Load the spaCy English model

The `en_core_web_sm` model includes a parser that can determine sentence boundaries. If the model is not installed, the next cell downloads it.

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Pipeline:", nlp.pipe_names)

## 3. Segmenter un texte avec spaCy

In [ ]:
text = (
    "This is the first sentence. "
    "This is another sentence. "
    "This is the last sentence."
)

doc = nlp(text)

for i, sent in enumerate(doc.sents, start=1):
    print(f"Sentence {i}: {sent.text}")

**Key idea:** `nlp(text)` creates a `Doc` object. The `doc.sents` property lets you iterate over the detected sentences.

## 4. Examiner les objets Span

In [ ]:
sentences = list(doc.sents)

print("Number of sentences:", len(sentences))
print("Second sentence:", sentences[1].text)
print("Token interval [start, end):", sentences[1].start, sentences[1].end)
print("Object type:", type(sentences[1]))

A sentence is represented by a `Span` object. The `start` and `end` indices delimit its tokens in the document; `end` is exclusive.

## 5. Repérer le début des phrases

In [ ]:
text = "This is a sentence. That is another sentence. Here is a third."
doc = nlp(text)

for token in doc:
    print(f"{token.text:>10} | is_sent_start={token.is_sent_start}")

`token.is_sent_start` is `True` for the first token of a sentence and `False` for other tokens. The first token in a document is normally a sentence start.

## 6. Pourquoi la segmentation peut être difficile

A period can end a sentence, but it can also occur in an abbreviation, a web address, or a decimal number. Language models use context to make better decisions.

In [ ]:
examples = [
    "Hello Mr. Smith, how are you today?",
    "The value is 3.26 dollars. The meeting is on Monday.",
    "Visit example.com for more information."
]

for text in examples:
    print("\nTEXT:", text)
    for sent in nlp(text).sents:
        print(" -", sent.text)

## 7. Segmentation personnalisée avec spaCy

You can add a task-specific rule. For example, treat a semicolon (`;`) as a sentence boundary. The component below is added **before** the parser so that the parser can take the boundary into account.

In [ ]:
from spacy.language import Language

@Language.component("semicolon_sentence_boundary")
def semicolon_sentence_boundary(doc):
    for token in doc[:-1]:
        if token.text == ";":
            doc[token.i + 1].is_sent_start = True
    return doc

# Add the component only once.
if "semicolon_sentence_boundary" not in nlp.pipe_names:
    nlp.add_pipe("semicolon_sentence_boundary", before="parser")

custom_text = "Management is doing things right; leadership is doing the right things."
custom_doc = nlp(custom_text)

for sent in custom_doc.sents:
    print("-", sent.text)

**Note:** custom rules should be evaluated on representative examples. A boundary that helps in one case may produce incorrect segmentation in another.

## 8. Utiliser NLTK

NLTK provides `sent_tokenize`, a convenient function for splitting text into sentences. It may require downloading tokenizer resources.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

for resource in ("punkt", "punkt_tab"):
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

example_text = (
    "Hello Mr. Smith, how are you doing today? "
    "The weather is great, and Python is awesome. "
    "The sky is pinkish-blue. You shouldn't eat cardboard."
)

for i, sentence in enumerate(sent_tokenize(example_text), start=1):
    print(f"{i}. {sentence}")

## 9. Comparer spaCy et NLTK

In [ ]:
text = (
    "Dr. Smith arrived at 10.30 a.m. "
    "He gave a short talk. Everyone enjoyed it!"
)

print("spaCy:")
for sent in nlp(text).sents:
    print("-", sent.text)

print("\nNLTK:")
for sent in sent_tokenize(text):
    print("-", sent)

The two libraries may produce different results depending on the text, models, and resources used. For a real project, inspect edge cases in your corpus.

## 10. Texte arabe : points d’attention

The English model `en_core_web_sm` is not suitable for Arabic linguistic analysis. For a basic segmentation example, `spacy.blank('ar')` provides an Arabic tokenizer without a parser. Sentence boundaries must be added explicitly, for example with `sentencizer`.

In [ ]:
arabic_nlp = spacy.blank("ar")
arabic_nlp.add_pipe("sentencizer")

arabic_text = "هذه هي الجملة الأولى. هذه هي الجملة الثانية؟ وهذه هي الجملة الثالثة!"
arabic_doc = arabic_nlp(arabic_text)

for i, sent in enumerate(arabic_doc.sents, start=1):
    print(f"{i}. {sent.text}")

This demonstration relies on punctuation. For Arabic morphological or syntactic analysis, choose a model and resources appropriate for the target language variety.

## 11. Tokenization simple avec `str.split()`

Python's `split()` method generally separates on whitespace. It does not account for linguistic structure and often leaves punctuation attached to words.

In [ ]:
text = "Sentence one. Sentence two! Is this sentence three?"
print(text.split())

## 12. Exercices

1. Write a text containing at least four sentences and print each sentence with `spaCy`.
2. Compare `spaCy` and `NLTK` on a sentence containing an abbreviation.
3. Test `I met Prof. Mohamed yesterday. We discussed NLP.` and inspect the segmentation.
4. Segment a short Arabic text with `spacy.blank('ar')` and `sentencizer`.
5. Explain why `str.split()` is not a reliable sentence segmentation method.

## 13. Summary

- Sentence segmentation identifies sentence units in text.
- `spaCy` uses an English model with a parser here; `doc.sents` yields sentences.
- `NLTK.sent_tokenize` provides another sentence segmentation approach.
- Abbreviations, numbers, URLs, and different languages create edge cases.
- Custom rules should be tested and adapted to the corpus.